# M7 - GEETA web app backend (FastAPI)

Implements PROJECT_BRIEF C7. Serves the kid-facing web app and the session API.

| Endpoint | What |
|----------|------|
| `GET /health` | liveness + model source (real/mock) |
| `POST /api/session` | create a session `{kid_name, problem_id?}` -> `{sid}` |
| `POST /api/session/{sid}/text` | kid text -> tutor reply + live metrics |
| `POST /api/session/{sid}/image` | canvas/photo PNG -> OCR -> kid-confirm prompt |
| `POST /api/session/{sid}/confirm` | kid confirms/corrects the OCR text |
| `POST /api/session/{sid}/audio` | voice -> ASR (graceful fallback if no local ASR) |
| `POST /api/session/{sid}/close` | end session -> parent PDF + dev JSON (dry-run delivery) |
| `GET /api/session/{sid}/report.pdf` | download the parent PDF |

**Privacy (hard):** session state is in-memory ONLY. Uploaded images are written
to a temp buffer, OCR'd, and deleted immediately - nothing is persisted beyond the
parent/dev report artefacts. No camera frames are stored (Phase 2, post-ethics).

**Model:** `get_model` uses the served llama-server (config `model.server_base_url`,
port 8080) when it is up, else the clearly-labelled `MockModel`. The web app runs on
port **8090** to avoid clashing with llama-server (8080) / LM Studio.

Run:  `.venv/bin/python -m uvicorn` is started by `run_webapp.sh` (see that script).


In [ ]:
# --- GEETA notebook-module bootstrap (idempotent; makes tools/*.ipynb importable as geeta.*) ---
import json as _json
import pathlib as _pathlib

_ROOT = _pathlib.Path.cwd()
if not (_ROOT / "tools" / "_nbmodule.ipynb").exists():      # run from project root or webapp/
    _ROOT = _ROOT.parent
_nb = _json.loads((_ROOT / "tools" / "_nbmodule.ipynb").read_text())
_src = "".join("".join(c["source"]) for c in _nb["cells"] if c["cell_type"] == "code")
exec(compile(_src, str(_ROOT / "tools" / "_nbmodule.ipynb"), "exec"), globals())
bootstrap(_ROOT)  # noqa: E501


In [ ]:
# NOTE: no `from __future__ import annotations` here. This notebook is exec'd into a
# plain dict by run_webapp.sh (not a real module), so PEP-563 string annotations would
# break Pydantic's forward-ref resolution for the request models below.
import os
import time
import uuid
import shutil
import tempfile
from pathlib import Path
from typing import Any, Optional

from geeta.model_client import load_config, get_model, ModelClient, MockModel
from geeta.tutor_engine import TutorSession

CFG = load_config(_ROOT)
OUT = Path(CFG["paths"]["outputs"])
WEBAPP_DIR = _ROOT / "webapp"
TMP_IMG = Path(tempfile.mkdtemp(prefix="geeta_img_"))   # transient OCR buffer only

# live tutoring sessions stay in memory; when one ends only its LEARNING
# SUMMARY + parent PDF are kept, in the local portal store (owner decision
# 2026-10-13 - see tools/portal_store.ipynb). No chat text, photos or camera data.
SESSIONS: dict[str, dict] = {}

from geeta.portal_store import PortalStore, summarise_session
# GEETA_PORTAL_DIR lets tests run against a throwaway store, never the real one
_PORTAL_DIR = Path(os.environ["GEETA_PORTAL_DIR"]) if os.environ.get("GEETA_PORTAL_DIR") else None
STORE = PortalStore((_PORTAL_DIR or _ROOT / "data" / "portal") / "geeta_portal.db",
                    (_PORTAL_DIR / "reports") if _PORTAL_DIR else OUT / "portal")


def _model_source() -> str:
    base = CFG["model"]["server_base_url"]
    c = ModelClient(base)
    return "real" if c.is_alive() else ("mock" if CFG["model"].get("allow_mock_model") else "none")


In [ ]:
import json
import collections
import posixpath
import threading
import urllib.parse

from fastapi import FastAPI, UploadFile, File, HTTPException, Request
from fastapi.responses import FileResponse, JSONResponse, RedirectResponse
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel, Field

app = FastAPI(title="GEETA 4B web app", version=CFG["project"]["version"],
              docs_url=None, redoc_url=None, openapi_url=None)


# ---- security (2026-10-13 audit) ---------------------------------------------------
class RateLimiter:
    """Sliding-window counters in memory, e.g. 'at most 20 messages a minute'.
    Bounded: old keys are swept so an attacker can't grow it forever."""

    def __init__(self, max_keys: int = 50_000):
        self._hits: dict[str, collections.deque] = {}
        self._lock = threading.Lock()
        self._max_keys = max_keys

    def hit(self, key: str, limit: int, window_s: float, count: bool = True) -> bool:
        """True if allowed. count=False only checks without recording."""
        now = time.time()
        with self._lock:
            q = self._hits.get(key)
            if q is None:
                if len(self._hits) >= self._max_keys:
                    self._sweep(now)
                q = self._hits[key] = collections.deque()
            while q and now - q[0] > window_s:
                q.popleft()
            if len(q) >= limit:
                return False
            if count:
                q.append(now)
            return True

    def record(self, key: str) -> None:
        with self._lock:
            self._hits.setdefault(key, collections.deque()).append(time.time())

    def clear(self, key: str) -> None:
        with self._lock:
            self._hits.pop(key, None)

    def _sweep(self, now: float) -> None:
        for k in [k for k, q in self._hits.items() if not q or now - q[-1] > 3600]:
            del self._hits[k]


class BodySizeLimit:
    """Refuse oversized requests before reading them (a 200 MB 'photo' used to be
    accepted and buffered in full). Also stops chunked bodies mid-stream."""

    def __init__(self, app, max_bytes: int):
        self.app, self.max_bytes = app, max_bytes

    async def __call__(self, scope, receive, send):
        if scope["type"] != "http":
            return await self.app(scope, receive, send)
        for k, v in scope.get("headers", []):
            if k == b"content-length":
                try:
                    too_big = int(v) > self.max_bytes
                except ValueError:
                    too_big = True
                if too_big:
                    resp = JSONResponse({"detail": "That's too big to send."}, status_code=413)
                    return await resp(scope, receive, send)
        seen = 0

        async def limited_receive():
            nonlocal seen
            msg = await receive()
            if msg["type"] == "http.request":
                seen += len(msg.get("body", b""))
                if seen > self.max_bytes:
                    raise HTTPException(413, "too big")
            return msg
        return await self.app(scope, limited_receive, send)


app.add_middleware(BodySizeLimit, max_bytes=16 * 1024 * 1024)

LIMITS = RateLimiter()
# (limit, window seconds)
LIMIT_LOGIN_ACCOUNT = (10, 15 * 60)   # wrong passwords per account, from anywhere
LIMIT_LOGIN_IP_ACCOUNT = (5, 15 * 60)
LIMIT_LOGIN_IP = (30, 15 * 60)
LIMIT_PIN_KID = (10, 15 * 60)         # wrong PINs per child, from anywhere
LIMIT_PIN_IP_KID = (5, 15 * 60)
LIMIT_MESSAGES = (20, 60)             # chat messages per person per minute
LIMIT_IMAGES = (6, 60)                # photo/PDF uploads per person per minute
LIMIT_SESSIONS = (10, 60)             # new tutoring sessions per person per minute
LIMIT_2FA_USER = (5, 15 * 60)         # wrong 2-step codes per account
MAX_LIVE_SESSIONS = 300
SESSION_IDLE_S = 3 * 3600
MAX_IMAGE_PIXELS = 40_000_000         # bigger "photos" are decompression bombs
MAX_IMAGE_SIDE = 2048                 # what the model needs; also strips metadata


def _limited(key: str, rule: tuple, what: str = "Slow down a little") -> None:
    if not LIMITS.hit(key, *rule):
        raise HTTPException(429, f"{what} - please wait a moment and try again.")

AUTH_COOKIE = "geeta_auth"      # a grown-up (parent / tutor / admin)
DEVICE_COOKIE = "geeta_device"  # this computer/tablet is switched on for kid sign-in
KID_COOKIE = "geeta_kid"        # a kid signed in with picture + PIN (tutoring only)



class TextIn(BaseModel):
    text: str = Field(max_length=2000)


class ConfirmIn(BaseModel):
    text: str = Field(max_length=2000)


class AttentionIn(BaseModel):
    # Privacy contract (session 13, user-approved + parent-consented): the
    # browser's on-device face detector NEVER sends a frame, image, or video
    # anywhere - only this short derived label crosses the network.
    state: str = Field(max_length=16)   # "engaged" | "away"


class SessionIn(BaseModel):
    kid_id: Optional[int] = None      # ignored for a signed-in kid (always themselves)
    problem_id: Optional[str] = Field(default=None, max_length=80)


class LoginIn(BaseModel):
    email: str = Field(max_length=254)
    password: str = Field(max_length=128)
    expect: Optional[str] = Field(default=None, max_length=10)   # "parent" | "staff"


class KidLoginIn(BaseModel):
    kid_id: int
    pin: str = Field(max_length=8)


class PinIn(BaseModel):
    pin: str = Field(max_length=8)


class ResetPwIn(BaseModel):
    password: str = Field(max_length=128)


class CodeIn(BaseModel):
    code: str = Field(max_length=20)


class LoginCodeIn(BaseModel):
    pending: str = Field(max_length=100)
    code: str = Field(max_length=20)


class TwoStepOffIn(BaseModel):
    password: str = Field(max_length=128)
    code: str = Field(max_length=20)


class SetupIn(BaseModel):
    name: str = Field(max_length=80)
    email: str = Field(max_length=254)
    password: str = Field(max_length=128)


class PasswordIn(BaseModel):
    current: str = Field(max_length=128)
    new: str = Field(max_length=128)


class KidIn(BaseModel):
    nickname: str = Field(max_length=40)
    year_level: Optional[int] = None
    parent_id: Optional[int] = None
    tutor_id: Optional[int] = None


class WorldIn(BaseModel):
    world: str = Field(max_length=16)


class PilotScoreIn(BaseModel):
    phase: str = Field(max_length=4)
    score: int
    out_of: int


class KidPatch(BaseModel):
    study_code: Optional[str] = Field(default=None, max_length=12)
    world: Optional[str] = Field(default=None, max_length=16)
    avatar: Optional[str] = Field(default=None, max_length=8)
    nickname: Optional[str] = Field(default=None, max_length=40)
    year_level: Optional[int] = None
    parent_id: Optional[int] = None
    tutor_id: Optional[int] = None


class UserIn(BaseModel):
    name: str = Field(max_length=80)
    email: str = Field(max_length=254)
    role: str = Field(max_length=10)
    password: str = Field(max_length=128)
    is_admin: bool = False


# ---- auth helpers ------------------------------------------------------------
def _client_ip(request: Request) -> str:
    return (request.headers.get("cf-connecting-ip")
            or (request.client.host if request.client else "?"))


def _is_direct_local(request: Request) -> bool:
    """True only for a browser on THIS Mac talking straight to uvicorn - not
    via a tunnel/proxy (those also connect from 127.0.0.1 but add headers)."""
    proxied = any(h in request.headers for h in
                  ("x-forwarded-for", "cf-connecting-ip", "x-real-ip", "forwarded", "cf-ray"))
    host = (request.headers.get("host") or "").split(":")[0]
    return (not proxied and request.client is not None
            and request.client.host in ("127.0.0.1", "::1")
            and host in ("127.0.0.1", "localhost"))


def _user(request: Request) -> Optional[dict]:
    return STORE.user_for_token(request.cookies.get(AUTH_COOKIE))


def _need_user(request: Request) -> dict:
    u = _user(request)
    if u is None:
        raise HTTPException(401, "please sign in")
    return u


def _log(request: Request, action: str, user_id=None, kid_id=None, detail: str = "") -> None:
    STORE.log_event(action, user_id=user_id, kid_id=kid_id, detail=detail, ip=_client_ip(request))


def _signed_in_response(request: Request, u: dict) -> JSONResponse:
    resp = JSONResponse({"ok": True, "user": u, "home": _home(u)})
    _set_auth_cookie(resp, request, STORE.new_token(u["id"]))
    STORE.drop_kid_token(request.cookies.get(KID_COOKIE))   # a grown-up took over this device
    resp.delete_cookie(KID_COOKIE, path="/")
    return resp


def _need_admin(request: Request) -> dict:
    u = _need_user(request)
    if not u["is_admin"]:
        raise HTTPException(403, "only the admin can do that")
    return u


def _need_kid(request: Request, kid_id: int) -> tuple[dict, dict]:
    u = _need_user(request)
    if not STORE.can_see_kid(u, kid_id):
        raise HTTPException(404, "child not found")
    return u, STORE.get_kid(kid_id)


def _is_https(request: Request) -> bool:
    """Behind the Cloudflare Tunnel uvicorn sees plain http from 127.0.0.1;
    the visitor's real scheme arrives in these headers."""
    return (request.url.scheme == "https"
            or request.headers.get("x-forwarded-proto", "").lower() == "https"
            or '"https"' in request.headers.get("cf-visitor", ""))


def _set_cookie(resp, request: Request, name: str, token: str, max_age: int) -> None:
    https = _is_https(request)
    resp.set_cookie(name, token, max_age=max_age, httponly=True,
                    samesite="lax", secure=https, path="/")


def _set_auth_cookie(resp, request: Request, token: str) -> None:
    _set_cookie(resp, request, AUTH_COOKIE, token, 14 * 86400)


def _kid(request: Request) -> Optional[dict]:
    return STORE.kid_for_token(request.cookies.get(KID_COOKIE))


def _home(user: dict) -> str:
    if user["is_admin"]:
        return "/admin.html"
    return "/tutor.html" if user["role"] == "tutor" else "/parent.html"


def _owner_key(request: Request) -> str:
    """Who owns a live tutoring session: a grown-up, or a signed-in kid."""
    u = _user(request)
    if u is not None:
        return f"u{u['id']}"
    k = _kid(request)
    if k is not None:
        return f"k{k['id']}"
    raise HTTPException(401, "please sign in")


def _same_origin(request: Request) -> bool:
    """CSRF guard for anything that changes data. A browser always says where
    a request came from (Origin, or Sec-Fetch-Site); another website - even
    another *.trycloudflare.com tunnel - must not act as a signed-in user."""
    origin = request.headers.get("origin")
    if origin:
        o = urllib.parse.urlsplit(origin)
        hosts = {request.headers.get("host", ""), request.headers.get("x-forwarded-host", "")}
        return o.netloc in hosts and o.netloc != ""
    site = request.headers.get("sec-fetch-site")
    return site in (None, "same-origin", "none")   # None: non-browser client (curl, tests)


# Pages that need someone signed in; the API checks per route. Static assets
# (css/js/images) carry no data and stay public so the sign-in pages can load.
_PAGE_ROLE = {"/": "learn", "/index.html": "learn", "/portal.html": "home",
              "/admin.html": "admin", "/tutor.html": "tutor", "/parent.html": "parent",
              "/pilot-test.html": "tutor"}
_PROTECTED_PAGES = tuple(_PAGE_ROLE)


# Only these files are web pages; everything else in webapp/ (the server
# notebook, run scripts) must never be downloadable, least of all via a tunnel.
_PUBLIC_FILES = ("/login.html", "/parent-login.html", "/staff-login.html", "/kid.html",
                 "/privacy.html", "/favicon.ico")
MAX_UPLOAD_BYTES = 15 * 1024 * 1024   # a phone photo is ~2-6 MB


def _page_redirect(request: Request, path: str) -> Optional[str]:
    """Where to send someone who opened a page that isn't theirs (None = allowed)."""
    need = _PAGE_ROLE[path]
    u = _user(request)
    if need == "learn":
        if u is not None or _kid(request) is not None:
            return None
        return "/kid.html" if request.cookies.get(DEVICE_COOKIE) else "/login.html"
    if u is None:
        return "/login.html"
    home = _home(u)
    if need == "home":
        return home
    ok = ((need == "admin" and u["is_admin"]) or (need == "tutor" and u["role"] == "tutor")
          or (need == "parent" and u["role"] == "parent"))
    return None if ok else home


_CSP = ("default-src 'self'; "
        "script-src 'self' https://cdn.jsdelivr.net 'wasm-unsafe-eval'; "      # focus helper (MediaPipe)
        "style-src 'self' 'unsafe-inline'; img-src 'self' data: blob:; media-src 'self' blob:; "
        "connect-src 'self' https://cdn.jsdelivr.net https://storage.googleapis.com; "
        "worker-src 'self' blob:; font-src 'self'; object-src 'none'; base-uri 'none'; "
        "form-action 'self'; frame-ancestors 'none'")


@app.middleware("http")
async def _auth_gate(request: Request, call_next):
    path = request.url.path
    # "/static/../geeta_server.ipynb" and %2e%2e tricks: refuse anything that
    # isn't already a clean path
    decoded = urllib.parse.unquote(request.scope.get("raw_path", b"").decode("latin-1") or path)
    if ".." in decoded or "\\" in decoded or posixpath.normpath(path) != path.rstrip("/") and path != "/":
        return JSONResponse({"detail": "Not Found"}, status_code=404)
    if not (path.startswith("/api/") or path.startswith("/static/") or path == "/health"
            or path in _PROTECTED_PAGES or path in _PUBLIC_FILES):
        return JSONResponse({"detail": "Not Found"}, status_code=404)
    if request.method in ("POST", "PUT", "PATCH", "DELETE") and not _same_origin(request):
        STORE.log_event("blocked_cross_site", detail=f"{request.method} {path}"[:120], ip=_client_ip(request))
        return JSONResponse({"detail": "request blocked: it didn't come from GEETA's own pages"},
                            status_code=403)
    if path in _PROTECTED_PAGES:
        where = _page_redirect(request, path)
        if where:
            if path == "/portal.html" and request.url.query:     # keep ?kid=N
                where += "?" + request.url.query
            return RedirectResponse(where, status_code=302)
    resp = await call_next(request)
    resp.headers.setdefault("X-Content-Type-Options", "nosniff")
    resp.headers.setdefault("X-Frame-Options", "DENY")
    resp.headers.setdefault("Referrer-Policy", "same-origin")
    resp.headers.setdefault("Permissions-Policy", "geolocation=(), payment=(), usb=(), microphone=(self), camera=(self)")
    resp.headers.setdefault("Content-Security-Policy", _CSP)
    resp.headers.setdefault("Cross-Origin-Opener-Policy", "same-origin")
    resp.headers.setdefault("Cross-Origin-Resource-Policy", "same-origin")
    if _is_https(request):
        resp.headers.setdefault("Strict-Transport-Security", "max-age=31536000")
    if path.startswith("/api/") or path in _PROTECTED_PAGES or path in _PUBLIC_FILES:
        # pages must never come from cache - or a signed-out browser could
        # show a stale page without going through the sign-in redirect
        resp.headers["Cache-Control"] = "no-store"
    return resp


def _get(sid: str, request: Request) -> dict:
    owner = _owner_key(request)
    s = SESSIONS.get(sid)
    if s is None or s["owner"] != owner:
        raise HTTPException(404, "unknown session")
    s["last"] = time.time()
    return s


def _sweep_sessions() -> None:
    """Save + drop tutoring sessions nobody has touched for a few hours."""
    now = time.time()
    for sid, s in list(SESSIONS.items()):
        if now - s.get("last", s["created"]) > SESSION_IDLE_S:
            if not s.get("report") and s.get("sess") is not None and _worth_saving(s["sess"]):
                _safe_reply(lambda: _finish_session(sid))
            SESSIONS.pop(sid, None)


def _metrics_view(sess: TutorSession) -> dict:
    """Lightweight live metrics for the UI (no close() - session continues)."""
    from geeta.reports import derive_metrics
    m = derive_metrics(sess, CFG)
    out = {k: m.get(k) for k in
           ("solved", "hints_used", "max_hint_depth", "reengagements",
            "attempts", "confidence", "engagement")}
    # celebrate only on the turn a question gets solved - `solved` stays true for
    # the rest of the session, which made every later reply set off confetti
    n = sum(1 for e in sess.events if e.kind == "solved")
    out["just_solved"] = n > getattr(sess, "_ui_solves_seen", 0)
    sess._ui_solves_seen = n
    return out


def _finish_session(sid: str) -> dict:
    """End a tutoring session: keep its learning summary + parent PDF in the
    kid's history, then drop the live session (chat/events) from memory."""
    s = SESSIONS[sid]
    if s.get("report"):
        return s["report"]
    from geeta.reports import derive_metrics, derive_grade, deliver_parent_report
    sess = s["sess"]
    closed = sess.close()
    m = derive_metrics(closed, CFG)
    summary = summarise_session(closed, m, derive_grade(m, CFG))
    kid_id = s["kid_id"]
    row_id = STORE.save_session(kid_id, summary, None)
    # technical tally per session: counts only, never text
    ev = closed.get("events") or []
    kinds = [e.get("kind") for e in ev]
    exact = sum(1 for e in ev if e.get("kind") == "select" and e.get("item_kind") in ("pattern", "words", "arith"))
    model_items = sum(1 for e in ev if e.get("kind") == "select" and e.get("item_kind") == "other")
    if not any(e.get("kind") == "select" for e in ev) and "verifier" in kinds:
        model_items += 1
    STORE.save_session_tech(row_id, {
        "exact_items": exact, "model_items": model_items,
        "gate_blocked": kinds.count("gate_blocked"), "safe_fallbacks": kinds.count("gate_fallback"),
        "leak_blocks": kinds.count("guard"), "photos": kinds.count("ocr"),
        "verifier_failed": sum(1 for e in ev if e.get("kind") == "verifier" and not e.get("ok")),
        "wrong_answer_risk": kinds.count("gate_fallback")})
    try:
        manifest = deliver_parent_report(closed, CFG, STORE.kid_report_dir(kid_id),
                                         history=STORE.progress(kid_id))
        pdf = Path(manifest["pdf"])
        STORE.set_session_pdf(row_id, pdf.name)
        pdf_url = f"/api/reports/{row_id}.pdf"
    except Exception:
        pdf_url = None            # the learning summary is still saved
    s["report"] = {"session_id": row_id, "pdf_url": pdf_url}
    s["sess"] = None          # live chat/events gone from memory too
    return s["report"]


def _worth_saving(sess: TutorSession) -> bool:
    return bool(sess is not None and (sess.solved or sess._item_stats or sess.hint_idx))


def _safe_reply(fn):
    """Run a TutorSession call that hits the model server and fail kid-
    friendly, not with a raw 500 - found in the session-13 QA sweep: any
    upstream hiccup (llama-server transient error, a malformed upload) was
    propagating uncaught, and fetch() doesn't treat an HTTP error status as
    a rejection, so the frontend silently did nothing (no bubble, no error)."""
    try:
        return fn()
    except Exception:
        return ("Hmm, something didn't quite work on my end - can you try "
                "that again?")


# ---- auth + accounts -----------------------------------------------------------
@app.get("/health")
def health():
    return {"ok": True, "model_source": _model_source(), "version": CFG["project"]["version"]}


@app.get("/api/auth/status")
def auth_status(request: Request):
    needs = STORE.user_count() == 0
    u = _user(request)
    return {"needs_setup": needs, "setup_allowed_here": needs and _is_direct_local(request),
            "signed_in": u is not None, "home": _home(u) if u else None,
            "kid_device": bool(request.cookies.get(DEVICE_COOKIE)),
            "kid_signed_in": _kid(request) is not None}


@app.post("/api/auth/setup")
def auth_setup(body: SetupIn, request: Request):
    """First run only: create the admin (tutor) account - and only from a
    browser on this Mac, never through the public tunnel."""
    if STORE.user_count() > 0:
        raise HTTPException(409, "already set up - please sign in")
    _limited(f"setup|{_client_ip(request)}", (10, 600))
    if not _is_direct_local(request):
        raise HTTPException(403, "the first account can only be made on the GEETA computer itself")
    try:
        uid = STORE.create_user(body.email, body.name, "tutor", body.password, is_admin=True)
    except ValueError as e:
        raise HTTPException(400, str(e))
    _log(request, "admin_created", user_id=uid)
    resp = JSONResponse({"ok": True, "user": STORE.get_user(uid), "home": "/admin.html"})
    _set_auth_cookie(resp, request, STORE.new_token(uid))
    return resp


@app.post("/api/auth/login")
def auth_login(body: LoginIn, request: Request):
    email = body.email.strip().lower()
    ip = _client_ip(request)
    keys = [(f"login|acct|{email}", LIMIT_LOGIN_ACCOUNT), (f"login|ipacct|{ip}|{email}", LIMIT_LOGIN_IP_ACCOUNT),
            (f"login|ip|{ip}", LIMIT_LOGIN_IP)]
    if not all(LIMITS.hit(k, *rule, count=False) for k, rule in keys):
        raise HTTPException(429, "too many tries - please wait 15 minutes")
    u = STORE.verify_login(email, body.password)
    if u is None:
        for k, _ in keys:
            LIMITS.record(k)
        _log(request, "login_failed", detail=email[:80])
        if not LIMITS.hit(f"login|acct|{email}", *LIMIT_LOGIN_ACCOUNT, count=False):
            _log(request, "account_locked", detail=email[:80])
        raise HTTPException(401, "email or password is not right")
    LIMITS.clear(f"login|ipacct|{ip}|{email}")
    if body.expect == "parent" and u["role"] != "parent":
        raise HTTPException(403, "This is the parents' sign-in - tutors and the admin use the Tutors & admin page.")
    if body.expect == "staff" and u["role"] != "tutor":
        raise HTTPException(403, "This is the tutors' sign-in - parents use the Parents page.")
    if u["two_step"]:
        return {"needs_code": True, "pending": STORE.new_pending_2fa(u["id"])}
    _log(request, "login", user_id=u["id"])
    return _signed_in_response(request, u)


@app.post("/api/auth/login/code")
def auth_login_code(body: LoginCodeIn, request: Request):
    """Second step: the 6-digit authenticator code (or a recovery code)."""
    uid_ = STORE.pending_2fa_user(body.pending)
    if uid_ is None:
        raise HTTPException(401, "that took too long - please sign in again")
    key = f"2fa|{uid_}"
    if not LIMITS.hit(key, *LIMIT_2FA_USER, count=False):
        raise HTTPException(429, "too many wrong codes - please wait 15 minutes")
    if not STORE.totp_check(uid_, body.code):
        LIMITS.record(key)
        _log(request, "2step_failed", user_id=uid_)
        raise HTTPException(401, "that code isn't right - check your authenticator app")
    STORE.pending_2fa_user(body.pending, consume=True)
    LIMITS.clear(key)
    _log(request, "login", user_id=uid_, detail="with 2-step")
    return _signed_in_response(request, STORE.get_user(uid_))


@app.post("/api/auth/2step/start")
def two_step_start(request: Request):
    u = _need_user(request)
    secret = STORE.totp_start(u["id"])
    label = urllib.parse.quote(f"GEETA:{u['email']}")
    return {"secret": secret, "uri": f"otpauth://totp/{label}?secret={secret}&issuer=GEETA&digits=6&period=30"}


@app.post("/api/auth/2step/confirm")
def two_step_confirm(body: CodeIn, request: Request):
    u = _need_user(request)
    codes = STORE.totp_confirm(u["id"], body.code)
    if codes is None:
        raise HTTPException(400, "that code isn't right - check the time on your phone and try the newest code")
    _log(request, "2step_on", user_id=u["id"])
    return {"ok": True, "recovery_codes": codes}


@app.post("/api/auth/2step/off")
def two_step_off(body: TwoStepOffIn, request: Request):
    u = _need_user(request)
    if STORE.verify_login(u["email"], body.password) is None or not STORE.totp_check(u["id"], body.code):
        raise HTTPException(400, "password or code isn't right")
    STORE.totp_disable(u["id"])
    _log(request, "2step_off", user_id=u["id"])
    return {"ok": True}


@app.post("/api/auth/logout")
def auth_logout(request: Request):
    u = _user(request)
    if u:
        _log(request, "logout", user_id=u["id"])
    STORE.drop_token(request.cookies.get(AUTH_COOKIE))
    resp = JSONResponse({"ok": True})
    resp.delete_cookie(AUTH_COOKIE, path="/")
    return resp


@app.get("/api/auth/me")
def auth_me(request: Request):
    u = _need_user(request)
    return {"user": u, "home": _home(u),
            "kid_device": STORE.user_for_device(request.cookies.get(DEVICE_COOKIE)) is not None}


@app.post("/api/auth/password")
def auth_password(body: PasswordIn, request: Request):
    u = _need_user(request)
    _limited(f"pwchange|{u['id']}", (5, 15 * 60), "Too many tries")
    if STORE.verify_login(u["email"], body.current) is None:
        raise HTTPException(400, "current password is not right")
    _log(request, "password_changed", user_id=u["id"])
    try:
        STORE.set_password(u["id"], body.new)
    except ValueError as e:
        raise HTTPException(400, str(e))
    resp = JSONResponse({"ok": True})
    _set_auth_cookie(resp, request, STORE.new_token(u["id"]))
    return resp


@app.get("/api/users")
def users_list(request: Request):
    _need_admin(request)
    return {"users": STORE.list_users()}


@app.post("/api/users")
def users_create(body: UserIn, request: Request):
    _need_admin(request)
    try:
        uid = STORE.create_user(body.email, body.name, body.role, body.password,
                                is_admin=body.is_admin and body.role == "tutor")
    except ValueError as e:
        raise HTTPException(400, str(e))
    _log(request, "account_created", user_id=uid, detail=f"{body.role} by admin")
    return {"user": STORE.get_user(uid)}


@app.delete("/api/users/{user_id}")
def users_delete(user_id: int, request: Request):
    me = _need_admin(request)
    if user_id == me["id"]:
        raise HTTPException(400, "you can't delete your own account")
    gone = STORE.get_user(user_id)
    STORE.delete_user(user_id)
    _log(request, "account_deleted", user_id=me["id"], detail=(gone or {}).get("email", "")[:80])
    return {"ok": True}


@app.post("/api/users/{user_id}/password")
def users_reset_password(user_id: int, body: ResetPwIn, request: Request):
    """Admin sets a new starting password for someone who forgot theirs."""
    _need_admin(request)
    if STORE.get_user(user_id) is None:
        raise HTTPException(404, "no such account")
    try:
        STORE.set_password(user_id, body.password)
    except ValueError as e:
        raise HTTPException(400, str(e))
    _log(request, "password_reset_by_admin", user_id=user_id)
    return {"ok": True}


@app.post("/api/users/{user_id}/2step/reset")
def users_reset_2step(user_id: int, request: Request):
    """Admin turns off 2-step for someone who lost their phone."""
    _need_admin(request)
    STORE.totp_disable(user_id)
    _log(request, "2step_reset_by_admin", user_id=user_id)
    return {"ok": True}


@app.get("/api/admin/security-log")
def admin_security_log(request: Request):
    _need_admin(request)
    return {"events": STORE.recent_events(150)}


@app.get("/api/admin/overview")
def admin_overview(request: Request):
    _need_admin(request)
    return {"counts": STORE.counts()}


# ---- kid sign-in (picture + PIN on a device a grown-up switched on) -----------------


@app.post("/api/kid/device")
def kid_device_on(request: Request):
    u = _need_user(request)
    resp = JSONResponse({"ok": True})
    _set_cookie(resp, request, DEVICE_COOKIE, STORE.new_device_token(u["id"]), 90 * 86400)
    _log(request, "kid_device_on", user_id=u["id"])
    return resp


@app.delete("/api/kid/device")
def kid_device_off(request: Request):
    _need_user(request)
    STORE.drop_device_token(request.cookies.get(DEVICE_COOKIE))
    resp = JSONResponse({"ok": True})
    resp.delete_cookie(DEVICE_COOKIE, path="/")
    return resp


def _device_user(request: Request) -> dict:
    d = STORE.user_for_device(request.cookies.get(DEVICE_COOKIE))
    if d is None:
        raise HTTPException(401, "a grown-up needs to switch on kid sign-in on this device first")
    return d


@app.get("/api/kid/roster")
def kid_roster(request: Request):
    d = _device_user(request)
    return {"kids": [{"id": k["id"], "nickname": k["nickname"], "avatar": k["avatar"],
                      "has_pin": bool(k["has_pin"])} for k in STORE.kids_for(d)]}


@app.post("/api/kid/login")
def kid_login(body: KidLoginIn, request: Request):
    d = _device_user(request)
    if not STORE.can_see_kid(d, body.kid_id):
        raise HTTPException(404, "child not found")
    keys = [(f"pin|kid|{body.kid_id}", LIMIT_PIN_KID),
            (f"pin|ipkid|{_client_ip(request)}|{body.kid_id}", LIMIT_PIN_IP_KID)]
    if not all(LIMITS.hit(k, *rule, count=False) for k, rule in keys):
        raise HTTPException(429, "Too many tries! Ask your grown-up, or wait 15 minutes. 🙂")
    if not STORE.verify_kid_pin(body.kid_id, body.pin):
        for k, _ in keys:
            LIMITS.record(k)
        _log(request, "kid_pin_failed", kid_id=body.kid_id)
        raise HTTPException(401, "Oops, that's not your secret number. Try again! 🙂")
    LIMITS.clear(keys[1][0])
    _log(request, "kid_signed_in", kid_id=body.kid_id)
    resp = JSONResponse({"ok": True})
    _set_cookie(resp, request, KID_COOKIE, STORE.new_kid_token(body.kid_id), 12 * 3600)
    # a grown-up signed in on the same device is signed out, so the kid
    # can't wander into the parent/tutor pages with the back button
    STORE.drop_token(request.cookies.get(AUTH_COOKIE))
    resp.delete_cookie(AUTH_COOKIE, path="/")
    return resp


@app.get("/api/kid/me")
def kid_me(request: Request):
    k = _kid(request)
    if k is None:
        raise HTTPException(401, "not signed in")
    return {"kid": {"id": k["id"], "nickname": k["nickname"], "avatar": k["avatar"]}}


@app.post("/api/kid/world")
def kid_world(body: WorldIn, request: Request):
    """A signed-in kid picks their world (teddy, car, ...) - saved on their profile."""
    k = _kid(request)
    if k is None:
        raise HTTPException(401, "not signed in")
    try:
        STORE.update_kid(k["id"], world=body.world)
    except ValueError as e:
        raise HTTPException(400, str(e))
    return {"ok": True}


@app.post("/api/kid/logout")
def kid_logout(request: Request):
    STORE.drop_kid_token(request.cookies.get(KID_COOKIE))
    resp = JSONResponse({"ok": True})
    resp.delete_cookie(KID_COOKIE, path="/")
    return resp


@app.post("/api/kids/{kid_id}/pin")
def kids_set_pin(kid_id: int, body: PinIn, request: Request):
    _need_kid(request, kid_id)
    try:
        STORE.set_kid_pin(kid_id, body.pin.strip())
    except ValueError as e:
        raise HTTPException(400, str(e))
    _log(request, "kid_pin_set", user_id=_user(request)["id"], kid_id=kid_id)
    return {"ok": True}


# ---- pilot study (tutors/admin only; research data is de-identified) -----------------
def _need_tutor(request: Request) -> dict:
    u = _need_user(request)
    if u["role"] != "tutor":
        raise HTTPException(403, "only tutors and the admin run the pilot check")
    return u


@app.get("/api/pilot/form")
def pilot_form(request: Request):
    """The 12-question pre/post check (parallel forms A+B) with its answer key."""
    _need_tutor(request)
    items = []
    for name in ("pre_form_a", "pre_form_b"):
        f = _ROOT / "data" / "eval" / "pilot_forms" / f"{name}.jsonl"
        for line in f.read_text().splitlines():
            if line.strip():
                d = json.loads(line)
                items.append({"q": d["question_text"].replace("  ", " ").strip(), "answer": d["answer"],
                              "year": d.get("year"), "skill": d.get("skill_id")})
    return {"items": items}


@app.post("/api/kids/{kid_id}/pilot-score")
def pilot_score(kid_id: int, body: PilotScoreIn, request: Request):
    u = _need_tutor(request)
    _need_kid(request, kid_id)
    try:
        STORE.set_pilot_score(kid_id, body.phase, body.score, body.out_of, u["id"])
    except ValueError as e:
        raise HTTPException(400, str(e))
    _log(request, "pilot_score", user_id=u["id"], kid_id=kid_id, detail=body.phase)
    return {"ok": True, "scores": STORE.pilot_scores(kid_id)}


@app.get("/api/pilot/export.csv")
def pilot_export(request: Request):
    import csv
    import io as _io
    from fastapi.responses import Response
    u = _need_admin(request)
    rows = STORE.research_export()
    buf = _io.StringIO()
    cols = ["study_code", "year_level", "sessions", "minutes", "questions_solved", "skills_practised",
            "skills_confident", "slip_categories", "pre_score", "pre_out_of", "post_score", "post_out_of", "gain_pp"]
    w = csv.DictWriter(buf, fieldnames=cols)
    w.writeheader()
    for r in rows:
        w.writerow(r)
    _log(request, "research_export", user_id=u["id"], detail=f"{len(rows)} children")
    return Response(buf.getvalue(), media_type="text/csv",
                    headers={"Content-Disposition": "attachment; filename=geeta_pilot_deidentified.csv"})


# ---- kids + learning history -------------------------------------------------------
@app.get("/api/kids")
def kids_list(request: Request):
    from geeta.portal_store import AVATARS
    return {"kids": STORE.kids_for(_need_user(request)), "avatars": AVATARS}


@app.post("/api/kids")
def kids_add(body: KidIn, request: Request):
    u = _need_user(request)
    if u["is_admin"]:
        parent_id, tutor_id = body.parent_id, body.tutor_id
    elif u["role"] == "parent":
        parent_id, tutor_id = u["id"], None
    else:
        parent_id, tutor_id = None, u["id"]      # only the admin links a child to a parent
    for uid, role in ((parent_id, "parent"), (tutor_id, "tutor")):
        if uid is not None:
            other = STORE.get_user(uid)
            if other is None or other["role"] != role:
                raise HTTPException(400, f"that {role} account doesn't exist")
    try:
        kid_id = STORE.add_kid(body.nickname, body.year_level, parent_id, tutor_id)
    except ValueError as e:
        raise HTTPException(400, str(e))
    _log(request, "child_added", user_id=u["id"], kid_id=kid_id)
    return {"kid": STORE.get_kid(kid_id)}


@app.patch("/api/kids/{kid_id}")
def kids_edit(kid_id: int, body: KidPatch, request: Request):
    u, kid = _need_kid(request, kid_id)
    changes = {k: v for k, v in body.model_dump(exclude_unset=True).items()}
    if not u["is_admin"] and ({"parent_id", "tutor_id"} & set(changes)):
        raise HTTPException(403, "only the admin can change who a child is linked to")
    if "study_code" in changes and u["role"] != "tutor":
        raise HTTPException(403, "only tutors and the admin set study codes")
    try:
        STORE.update_kid(kid_id, **changes)
    except ValueError as e:
        raise HTTPException(400, str(e))
    return {"kid": STORE.get_kid(kid_id)}


@app.delete("/api/kids/{kid_id}")
def kids_delete(kid_id: int, request: Request):
    u, kid = _need_kid(request, kid_id)
    if not (u["is_admin"] or kid["parent_id"] == u["id"]):
        raise HTTPException(403, "only the child's parent or the admin can delete their data")
    STORE.delete_kid(kid_id)
    _log(request, "child_deleted", user_id=u["id"], detail=kid["nickname"][:40])
    return {"ok": True}


@app.get("/api/kids/{kid_id}/progress")
def kids_progress(kid_id: int, request: Request):
    _, kid = _need_kid(request, kid_id)
    from geeta.worksheet import MISTAKE_TEXT
    prog = STORE.progress(kid_id)
    return {"kid": kid, "progress": prog, "pilot": STORE.pilot_scores(kid_id),
            "watch_text": [MISTAKE_TEXT.get(c, c) for c in prog["watch_for"]],
            "sessions": [{**r, "pdf_url": (f"/api/reports/{r['id']}.pdf" if r["pdf_name"] else None)}
                         for r in STORE.sessions_for(kid_id)]}


@app.get("/api/reports/{session_id}.pdf")
def stored_report(session_id: int, request: Request):
    row = STORE.get_session(session_id)
    if row is None:
        raise HTTPException(404, "report not found")
    _need_kid(request, row["kid_id"])
    p = STORE.kid_report_dir(row["kid_id"]) / "parent" / (row["pdf_name"] or "")
    if not row["pdf_name"] or not p.is_file():
        raise HTTPException(404, "report missing")
    return FileResponse(p, media_type="application/pdf", filename=p.name)


# ---- tutoring sessions ---------------------------------------------------------------
@app.post("/api/session")
def create_session(body: SessionIn, request: Request):
    owner = _owner_key(request)
    _limited(f"sess|{owner}", LIMIT_SESSIONS)
    _sweep_sessions()
    if len(SESSIONS) >= MAX_LIVE_SESSIONS:
        raise HTTPException(503, "GEETA is very busy right now - please try again in a few minutes.")
    if owner.startswith("k"):
        kid = _kid(request)                      # a signed-in kid is always themselves
    else:
        if body.kid_id is None:
            raise HTTPException(400, "pick a child first")
        _, kid = _need_kid(request, body.kid_id)
    # a session the kid walked away from still counts: save it before starting anew
    for old_sid, old in list(SESSIONS.items()):
        if old["owner"] == owner and not old.get("report"):
            if _worth_saving(old["sess"]):
                _safe_reply(lambda: _finish_session(old_sid))
            SESSIONS.pop(old_sid, None)
    sid = uuid.uuid4().hex[:12]
    problem = _load_problem(body.problem_id) if body.problem_id else None
    model = get_model(CFG, problem)
    profile = STORE.profile_for_tutor(kid["id"])
    buddy = {"teddy": "Bobo", "car": "Zoom", "space": "Cosmo", "dino": "Rexy",
             "unicorn": "Luna", "ocean": "Ollie"}.get(kid.get("world") or "", "GEETA")
    sess = TutorSession(CFG, model, kid_name=kid["nickname"], problem=problem, kid_profile=profile,
                        buddy=buddy)
    SESSIONS[sid] = {"sess": sess, "kid_id": kid["id"], "owner": owner,
                     "created": time.time(), "last": time.time(), "report": None}
    if problem:
        greeting = sess.feed_text("hi")
    elif profile.get("sessions") and profile.get("last_skill"):
        greeting = (f"Welcome back, {kid['nickname']}! 🌟 Last time we did "
                    f"{profile['last_skill'][0].lower() + profile['last_skill'][1:]}. "
                    "Show me today's homework - draw it, snap a photo, type it, or say it.")
    else:
        greeting = (f"Hi {kid['nickname']}! Show me your homework - draw it, "
                    "snap a photo, type it, or say it. I'll help you work it out yourself.")
    # bug found live-testing (session 13): this used to return the raw Python
    # class name ("ModelClient"/"MockModel"), but app.js's badge compares
    # against the literal string "real" (same convention as /health) - so a
    # genuinely real-backed session showed "practice mode" in the UI.
    return {"sid": sid, "model_source": "real" if isinstance(model, ModelClient) else "mock",
            "kid": {"id": kid["id"], "nickname": kid["nickname"], "avatar": kid.get("avatar"),
                    "world": kid.get("world")},
            "kid_mode": owner.startswith("k"), "greeting": greeting}


def _load_problem(problem_id: str) -> dict | None:
    """Load a problem from the ACARA bank by id (for the demo board)."""
    bank = _ROOT / "data" / "bank" / "acara_y1-3.jsonl"
    if not bank.exists():
        return None
    for line in bank.read_text().splitlines():
        if not line.strip():
            continue
        import json as _j
        d = _j.loads(line)
        if d.get("id") == problem_id:
            return d
    return None


def _live(s: dict) -> TutorSession:
    if s["sess"] is None:
        raise HTTPException(409, "this session has finished - start a new one")
    return s["sess"]


@app.post("/api/session/{sid}/text")
def feed_text(sid: str, body: TextIn, request: Request):
    s = _get(sid, request)
    _limited(f"msg|{s['owner']}", LIMIT_MESSAGES, "Whoa, that's a lot of messages")
    sess = _live(s)
    reply = _safe_reply(lambda: sess.feed_text(body.text))
    return {"reply": reply, "metrics": _metrics_view(sess)}


@app.post("/api/session/{sid}/image")
async def feed_image(sid: str, request: Request, file: UploadFile = File(...)):
    s = _get(sid, request)
    _limited(f"img|{s['owner']}", LIMIT_IMAGES, "That's a lot of photos")
    sess = _live(s)
    raw = await file.read(MAX_UPLOAD_BYTES + 1)
    if len(raw) > MAX_UPLOAD_BYTES:
        return {"reply": "That file is too big for me - can you take a smaller photo?",
                "metrics": _metrics_view(sess)}
    is_pdf = (file.content_type == "application/pdf") or (file.filename or "").lower().endswith(".pdf")
    # transient buffer: write, OCR, delete (privacy: nothing persisted)
    dest = TMP_IMG / f"{sid}_{uuid.uuid4().hex[:6]}.png"
    pdf_tmp = None
    try:
        if is_pdf:
            # real homework often arrives as a PDF scan/export, not just a photo -
            # reuse the same pypdfium2 render path as the C1 offline pipeline
            # (notebooks/02_pdf_to_json_pipeline.ipynb) to turn page 1 into a PNG.
            import pypdfium2 as pdfium
            pdf_tmp = TMP_IMG / f"{sid}_{uuid.uuid4().hex[:6]}.pdf"
            pdf_tmp.write_bytes(raw)
            doc = pdfium.PdfDocument(str(pdf_tmp))
            pw, ph = doc[0].get_size()                      # points; huge pages = render bomb
            scale = min(170 / 72.0, MAX_IMAGE_SIDE / max(pw, ph, 1))
            bmp = doc[0].render(scale=scale)
            bmp.to_pil().save(dest)
            doc.close()
            ok = True
        else:
            ok = _safe_image(raw, dest)
        if not ok:
            return {"reply": "Hmm, that doesn't look like a photo I can read - try taking a "
                             "picture of your homework, or type the question.",
                    "metrics": _metrics_view(sess)}
        try:
            reply = sess.feed_image(dest)
        except Exception:
            # QA sweep (session 13) found: a malformed/corrupt upload made
            # llama-server 400 the OCR call, ModelClient.raise_for_status()
            # propagated that uncaught all the way to a raw 500 - and since
            # fetch() doesn't reject on HTTP error statuses, the frontend
            # silently did nothing (typing indicator cleared, no message,
            # no error). Kid-friendly fallback instead; nothing here is a
            # crash the kid should ever see.
            reply = ("Hmm, I couldn't read that photo properly - can you try "
                     "taking it again, or type the question instead?")
    finally:
        dest.unlink(missing_ok=True)
        if pdf_tmp is not None:
            pdf_tmp.unlink(missing_ok=True)
    return {"reply": reply, "metrics": _metrics_view(sess)}


def _safe_image(raw: bytes, dest: Path) -> bool:
    """Accept only real, sanely-sized images. Re-encoding drops anything
    hidden in the file (metadata, trailing data) and shrinks big photos."""
    from PIL import Image
    import io as _io
    try:
        with Image.open(_io.BytesIO(raw)) as im:
            if im.format not in ("PNG", "JPEG", "WEBP", "GIF", "HEIF", "MPO") or \
                    im.width * im.height > MAX_IMAGE_PIXELS:
                return False
            im.load()
            im = im.convert("RGB")
            im.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE))
            im.save(dest, "PNG")
        return True
    except Exception:
        return False


@app.post("/api/session/{sid}/confirm")
def confirm_ocr(sid: str, body: ConfirmIn, request: Request):
    s = _get(sid, request)
    _limited(f"msg|{s['owner']}", LIMIT_MESSAGES, "Whoa, that's a lot of messages")
    sess = _live(s)
    reply = _safe_reply(lambda: sess.confirm_ocr(body.text))
    return {"reply": reply, "metrics": _metrics_view(sess)}


@app.post("/api/session/{sid}/attention")
def feed_attention(sid: str, body: AttentionIn, request: Request):
    """Privacy-safe engagement signal only - see AttentionIn. Returns a reply
    (a warm re-engagement nudge, possibly with a story) only when sustained
    distraction is detected; otherwise reply is None and the frontend adds
    no chat bubble."""
    s = _get(sid, request)
    if s["sess"] is None:
        return {"reply": None, "metrics": None}
    reply = _safe_reply(lambda: s["sess"].attention_signal(body.state))
    return {"reply": reply, "metrics": _metrics_view(s["sess"])}


@app.post("/api/session/{sid}/close")
def close_session(sid: str, request: Request):
    s = _get(sid, request)
    if s["sess"] is not None and not _worth_saving(s["sess"]) and not s.get("report"):
        return {"report": None, "saved": False}
    rep_ = _finish_session(sid)
    if s["owner"].startswith("k"):
        return {"report": None, "saved": True}   # reports are for grown-ups
    return {"report": rep_, "saved": True}


@app.get("/api/session/{sid}/report.pdf")
def report_pdf(sid: str, request: Request):
    s = _get(sid, request)
    if not s.get("report"):
        raise HTTPException(409, "session not closed yet")
    return stored_report(s["report"]["session_id"], request)


# Pages are served one by one from a fixed list; assets only from webapp/static.
# (The whole webapp/ folder used to be mounted, which let "/static/../" reach
# the server notebook.)
def _page(name: str):
    def handler():
        return FileResponse(WEBAPP_DIR / name, media_type="text/html")
    return handler


for _path, _file in [("/", "index.html"), ("/index.html", "index.html"), ("/portal.html", "portal.html"),
                     ("/admin.html", "admin.html"), ("/tutor.html", "tutor.html"), ("/parent.html", "parent.html"),
                     ("/login.html", "login.html"), ("/parent-login.html", "parent-login.html"),
                     ("/staff-login.html", "staff-login.html"), ("/kid.html", "kid.html"),
                     ("/privacy.html", "privacy.html"), ("/pilot-test.html", "pilot-test.html")]:
    app.add_api_route(_path, _page(_file), methods=["GET"], include_in_schema=False)

app.mount("/static", StaticFiles(directory=str(WEBAPP_DIR / "static")), name="static")


In [ ]:
# Run the server. Gated so importing geeta's webapp module never auto-starts it.
# run_webapp.sh sets GEETA_RUN_SERVER=1 and calls uvicorn on the app object.
import os as _os

if _os.environ.get("GEETA_RUN_SERVER") == "1" or __name__ == "__main__":
    import uvicorn
    _port = int(_os.environ.get("GEETA_PORT", "8090"))
    print(f"[GEETA] web app on http://127.0.0.1:{_port}  (model source: {_model_source()})")
    uvicorn.run(app, host="127.0.0.1", port=_port, log_level="info")
